In [105]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.utils import shuffle
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

# EDA

In [106]:
features = pd.read_csv('../data/features.txt', sep=r'\s+', header=None, names=['index', 'feature_name'])

X = pd.read_csv('../data/train/X_train.txt', sep=r'\s+', header=None)
y = pd.read_csv('../data/train/y_train.txt', sep=r'\s+', header=None, names=['activity'])

X_test = pd.read_csv("../data/test/X_test.txt", sep=r"\s+", header=None)
y_test = pd.read_csv("../data/test/y_test.txt", sep=r"\s+", header=None, names=["activity"])
subjects = pd.read_csv('../data/train/subject_train.txt', sep=r'\s+', header=None, names=['subject'])
activity_labels = pd.read_csv('../data/activity_labels.txt', sep=r'\s+', header=None, names=['index', 'activity_name'])

In [107]:
RANDOM_SEED = 42

In [108]:
X, y, subjects = shuffle(X, y, subjects, random_state=RANDOM_SEED)

In [114]:
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
X_test_scaled = scaler.transform(X_test)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['activity'] = y.activity

In [115]:
activity_map = {}
for i, name in enumerate(activity_labels.activity_name):
    activity_map[i+1] = name

pca_df['activity_name'] = pca_df['activity'].map(activity_map)

In [121]:
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

X_pca = pca.fit_transform(X_scaled)
X_test_pca = pca.transform(X_test_scaled)



In [122]:
# Label encoding
le = LabelEncoder()
y_encoded = le.fit_transform(y['activity']) 
y_test_encoded = le.transform(y_test['activity'])
y_encoded
y_test_encoded

array([4, 4, 4, ..., 1, 1, 1], dtype=int64)

## Train and Valdiation Split

In [123]:
from sklearn.model_selection import train_test_split

# First split: train + temp
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_encoded, test_size=0.15, random_state=42
)


## 5-layer MLP

In [125]:
import torch.nn as nn

class LargeModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.4),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.4),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [126]:
import os
os.environ["TORCH_DISABLE_DYNAMO"] = "1"

In [127]:
import torch

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test_encoded, dtype=torch.long)

## 4-Layer MLP

In [129]:
class SmallModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.4),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# Training model function

In [130]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score

def train(model=None, lr=5e-4, batch_size=64, epochs=30):
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size)

    best_val_acc = 0

    for epoch in range(epochs):
        # ---- TRAIN ----
        model.train()
        train_correct = 0
        train_total = 0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()

            preds = torch.argmax(outputs, dim=1)

            train_correct += (preds == yb).sum().item()
            train_total += yb.size(0)

        train_acc = train_correct / train_total

        # ---- VALIDATION ----
        model.eval()
        val_preds = []
        val_true = []

        with torch.no_grad():
            for xb, yb in val_loader:
                outputs = model(xb)
                pred = torch.argmax(outputs, dim=1)

                val_preds.extend(pred.cpu().numpy())
                val_true.extend(yb.cpu().numpy())

        val_acc = accuracy_score(val_true, val_preds)

        print(f"Epoch {epoch+1}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

        # ---- SAVE BEST MODEL ----
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict()

    return best_model_state, best_val_acc

## Largemodel Optimization

In [131]:
model = LargeModel(input_dim=X_train_t.shape[1], num_classes=6)

### Batch Size tested: 64, 128, 256

In [132]:
best_model_state, best_val_acc = train(model = model, lr=5e-4, batch_size=64, epochs=30)

Epoch 1, Train Acc: 0.8038, Val Acc: 0.9519
Epoch 2, Train Acc: 0.9448, Val Acc: 0.9655
Epoch 3, Train Acc: 0.9610, Val Acc: 0.9719
Epoch 4, Train Acc: 0.9664, Val Acc: 0.9701
Epoch 5, Train Acc: 0.9728, Val Acc: 0.9801
Epoch 6, Train Acc: 0.9738, Val Acc: 0.9791
Epoch 7, Train Acc: 0.9770, Val Acc: 0.9782
Epoch 8, Train Acc: 0.9768, Val Acc: 0.9782
Epoch 9, Train Acc: 0.9806, Val Acc: 0.9810
Epoch 10, Train Acc: 0.9794, Val Acc: 0.9791
Epoch 11, Train Acc: 0.9805, Val Acc: 0.9801
Epoch 12, Train Acc: 0.9802, Val Acc: 0.9764
Epoch 13, Train Acc: 0.9824, Val Acc: 0.9828
Epoch 14, Train Acc: 0.9819, Val Acc: 0.9828
Epoch 15, Train Acc: 0.9822, Val Acc: 0.9773
Epoch 16, Train Acc: 0.9827, Val Acc: 0.9791
Epoch 17, Train Acc: 0.9843, Val Acc: 0.9828
Epoch 18, Train Acc: 0.9850, Val Acc: 0.9837
Epoch 19, Train Acc: 0.9845, Val Acc: 0.9819
Epoch 20, Train Acc: 0.9848, Val Acc: 0.9801
Epoch 21, Train Acc: 0.9864, Val Acc: 0.9891
Epoch 22, Train Acc: 0.9838, Val Acc: 0.9837
Epoch 23, Train Acc

In [133]:
print("Best Validation Accuracy:", best_val_acc)

Best Validation Accuracy: 0.9891205802357208


In [134]:
best_model_state, best_val_acc = train(model = model, lr=5e-4, batch_size=128, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9870, Val Acc: 0.9819
Epoch 2, Train Acc: 0.9920, Val Acc: 0.9873
Epoch 3, Train Acc: 0.9930, Val Acc: 0.9810
Epoch 4, Train Acc: 0.9934, Val Acc: 0.9846
Epoch 5, Train Acc: 0.9926, Val Acc: 0.9773
Epoch 6, Train Acc: 0.9901, Val Acc: 0.9810
Epoch 7, Train Acc: 0.9936, Val Acc: 0.9755
Epoch 8, Train Acc: 0.9912, Val Acc: 0.9801
Epoch 9, Train Acc: 0.9938, Val Acc: 0.9828
Epoch 10, Train Acc: 0.9944, Val Acc: 0.9873
Epoch 11, Train Acc: 0.9946, Val Acc: 0.9828
Epoch 12, Train Acc: 0.9933, Val Acc: 0.9810
Epoch 13, Train Acc: 0.9931, Val Acc: 0.9773
Epoch 14, Train Acc: 0.9952, Val Acc: 0.9837
Epoch 15, Train Acc: 0.9931, Val Acc: 0.9846
Epoch 16, Train Acc: 0.9952, Val Acc: 0.9791
Epoch 17, Train Acc: 0.9947, Val Acc: 0.9791
Epoch 18, Train Acc: 0.9955, Val Acc: 0.9846
Epoch 19, Train Acc: 0.9928, Val Acc: 0.9873
Epoch 20, Train Acc: 0.9942, Val Acc: 0.9855
Epoch 21, Train Acc: 0.9923, Val Acc: 0.9864
Epoch 22, Train Acc: 0.9954, Val Acc: 0.9819
Epoch 23, Train Acc

In [135]:
best_model_state, best_val_acc = train(model = model, lr=5e-4, batch_size=256, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9942, Val Acc: 0.9801
Epoch 2, Train Acc: 0.9942, Val Acc: 0.9855
Epoch 3, Train Acc: 0.9976, Val Acc: 0.9855
Epoch 4, Train Acc: 0.9974, Val Acc: 0.9837
Epoch 5, Train Acc: 0.9968, Val Acc: 0.9864
Epoch 6, Train Acc: 0.9963, Val Acc: 0.9864
Epoch 7, Train Acc: 0.9971, Val Acc: 0.9855
Epoch 8, Train Acc: 0.9946, Val Acc: 0.9846
Epoch 9, Train Acc: 0.9965, Val Acc: 0.9864
Epoch 10, Train Acc: 0.9984, Val Acc: 0.9855
Epoch 11, Train Acc: 0.9974, Val Acc: 0.9864
Epoch 12, Train Acc: 0.9971, Val Acc: 0.9828
Epoch 13, Train Acc: 0.9971, Val Acc: 0.9864
Epoch 14, Train Acc: 0.9968, Val Acc: 0.9846
Epoch 15, Train Acc: 0.9976, Val Acc: 0.9819
Epoch 16, Train Acc: 0.9978, Val Acc: 0.9837
Epoch 17, Train Acc: 0.9965, Val Acc: 0.9864
Epoch 18, Train Acc: 0.9970, Val Acc: 0.9855
Epoch 19, Train Acc: 0.9968, Val Acc: 0.9873
Epoch 20, Train Acc: 0.9939, Val Acc: 0.9846
Epoch 21, Train Acc: 0.9965, Val Acc: 0.9819
Epoch 22, Train Acc: 0.9963, Val Acc: 0.9801
Epoch 23, Train Acc

###  Learning rate tested: 0.0001, 0.0005, 0.001, 0.005, 0.01

In [136]:
best_model_state, best_val_acc = train(model = model, lr=1e-3, batch_size=256, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9931, Val Acc: 0.9773
Epoch 2, Train Acc: 0.9917, Val Acc: 0.9801
Epoch 3, Train Acc: 0.9934, Val Acc: 0.9801
Epoch 4, Train Acc: 0.9922, Val Acc: 0.9846
Epoch 5, Train Acc: 0.9933, Val Acc: 0.9791
Epoch 6, Train Acc: 0.9947, Val Acc: 0.9900
Epoch 7, Train Acc: 0.9946, Val Acc: 0.9873
Epoch 8, Train Acc: 0.9941, Val Acc: 0.9846
Epoch 9, Train Acc: 0.9947, Val Acc: 0.9873
Epoch 10, Train Acc: 0.9946, Val Acc: 0.9855
Epoch 11, Train Acc: 0.9942, Val Acc: 0.9782
Epoch 12, Train Acc: 0.9949, Val Acc: 0.9864
Epoch 13, Train Acc: 0.9934, Val Acc: 0.9828
Epoch 14, Train Acc: 0.9918, Val Acc: 0.9764
Epoch 15, Train Acc: 0.9893, Val Acc: 0.9801
Epoch 16, Train Acc: 0.9891, Val Acc: 0.9791
Epoch 17, Train Acc: 0.9930, Val Acc: 0.9855
Epoch 18, Train Acc: 0.9914, Val Acc: 0.9864
Epoch 19, Train Acc: 0.9920, Val Acc: 0.9773
Epoch 20, Train Acc: 0.9944, Val Acc: 0.9846
Epoch 21, Train Acc: 0.9941, Val Acc: 0.9801
Epoch 22, Train Acc: 0.9928, Val Acc: 0.9900
Epoch 23, Train Acc

In [137]:
best_model_state, best_val_acc = train(model = model, lr=1e-2, batch_size=256, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9077, Val Acc: 0.9220
Epoch 2, Train Acc: 0.9542, Val Acc: 0.9574
Epoch 3, Train Acc: 0.9670, Val Acc: 0.9701
Epoch 4, Train Acc: 0.9611, Val Acc: 0.9710
Epoch 5, Train Acc: 0.9659, Val Acc: 0.9637
Epoch 6, Train Acc: 0.9600, Val Acc: 0.9646
Epoch 7, Train Acc: 0.9648, Val Acc: 0.9655
Epoch 8, Train Acc: 0.9662, Val Acc: 0.9728
Epoch 9, Train Acc: 0.9642, Val Acc: 0.9601
Epoch 10, Train Acc: 0.9642, Val Acc: 0.9438
Epoch 11, Train Acc: 0.9686, Val Acc: 0.9710
Epoch 12, Train Acc: 0.9739, Val Acc: 0.9674
Epoch 13, Train Acc: 0.9762, Val Acc: 0.9665
Epoch 14, Train Acc: 0.9710, Val Acc: 0.9764
Epoch 15, Train Acc: 0.9691, Val Acc: 0.9592
Epoch 16, Train Acc: 0.9672, Val Acc: 0.9447
Epoch 17, Train Acc: 0.9683, Val Acc: 0.9719
Epoch 18, Train Acc: 0.9661, Val Acc: 0.9665
Epoch 19, Train Acc: 0.9712, Val Acc: 0.9773
Epoch 20, Train Acc: 0.9698, Val Acc: 0.9193
Epoch 21, Train Acc: 0.9690, Val Acc: 0.9655
Epoch 22, Train Acc: 0.9659, Val Acc: 0.9411
Epoch 23, Train Acc

In [138]:
best_model_state, best_val_acc = train(model = model, lr=5e-3, batch_size=256, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9747, Val Acc: 0.9529
Epoch 2, Train Acc: 0.9800, Val Acc: 0.9746
Epoch 3, Train Acc: 0.9787, Val Acc: 0.9746
Epoch 4, Train Acc: 0.9805, Val Acc: 0.9710
Epoch 5, Train Acc: 0.9805, Val Acc: 0.9737
Epoch 6, Train Acc: 0.9786, Val Acc: 0.9637
Epoch 7, Train Acc: 0.9765, Val Acc: 0.9710
Epoch 8, Train Acc: 0.9786, Val Acc: 0.9737
Epoch 9, Train Acc: 0.9797, Val Acc: 0.9402
Epoch 10, Train Acc: 0.9758, Val Acc: 0.9683
Epoch 11, Train Acc: 0.9766, Val Acc: 0.9674
Epoch 12, Train Acc: 0.9813, Val Acc: 0.9665
Epoch 13, Train Acc: 0.9800, Val Acc: 0.9574
Epoch 14, Train Acc: 0.9794, Val Acc: 0.9719
Epoch 15, Train Acc: 0.9797, Val Acc: 0.9628
Epoch 16, Train Acc: 0.9790, Val Acc: 0.9737
Epoch 17, Train Acc: 0.9811, Val Acc: 0.9574
Epoch 18, Train Acc: 0.9795, Val Acc: 0.9293
Epoch 19, Train Acc: 0.9773, Val Acc: 0.9492
Epoch 20, Train Acc: 0.9784, Val Acc: 0.9710
Epoch 21, Train Acc: 0.9798, Val Acc: 0.9746
Epoch 22, Train Acc: 0.9830, Val Acc: 0.9719
Epoch 23, Train Acc

In [139]:
best_model_state, best_val_acc = train(model = model, lr=1e-4, batch_size=256, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9835, Val Acc: 0.9837
Epoch 2, Train Acc: 0.9867, Val Acc: 0.9828
Epoch 3, Train Acc: 0.9854, Val Acc: 0.9828
Epoch 4, Train Acc: 0.9866, Val Acc: 0.9837
Epoch 5, Train Acc: 0.9870, Val Acc: 0.9828
Epoch 6, Train Acc: 0.9872, Val Acc: 0.9828
Epoch 7, Train Acc: 0.9878, Val Acc: 0.9837
Epoch 8, Train Acc: 0.9902, Val Acc: 0.9846
Epoch 9, Train Acc: 0.9891, Val Acc: 0.9846
Epoch 10, Train Acc: 0.9904, Val Acc: 0.9828
Epoch 11, Train Acc: 0.9891, Val Acc: 0.9810
Epoch 12, Train Acc: 0.9901, Val Acc: 0.9837
Epoch 13, Train Acc: 0.9899, Val Acc: 0.9846
Epoch 14, Train Acc: 0.9898, Val Acc: 0.9846
Epoch 15, Train Acc: 0.9898, Val Acc: 0.9837
Epoch 16, Train Acc: 0.9912, Val Acc: 0.9837
Epoch 17, Train Acc: 0.9899, Val Acc: 0.9810
Epoch 18, Train Acc: 0.9915, Val Acc: 0.9837
Epoch 19, Train Acc: 0.9918, Val Acc: 0.9846
Epoch 20, Train Acc: 0.9899, Val Acc: 0.9846
Epoch 21, Train Acc: 0.9912, Val Acc: 0.9828
Epoch 22, Train Acc: 0.9917, Val Acc: 0.9828
Epoch 23, Train Acc

In [140]:
best_model_state, best_val_acc = train(model = model, lr=1e-3, batch_size=64, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9832, Val Acc: 0.9810
Epoch 2, Train Acc: 0.9866, Val Acc: 0.9801
Epoch 3, Train Acc: 0.9886, Val Acc: 0.9810
Epoch 4, Train Acc: 0.9869, Val Acc: 0.9801
Epoch 5, Train Acc: 0.9853, Val Acc: 0.9810
Epoch 6, Train Acc: 0.9838, Val Acc: 0.9791
Epoch 7, Train Acc: 0.9880, Val Acc: 0.9764
Epoch 8, Train Acc: 0.9872, Val Acc: 0.9773
Epoch 9, Train Acc: 0.9882, Val Acc: 0.9810
Epoch 10, Train Acc: 0.9882, Val Acc: 0.9801
Epoch 11, Train Acc: 0.9859, Val Acc: 0.9819
Epoch 12, Train Acc: 0.9848, Val Acc: 0.9819
Epoch 13, Train Acc: 0.9886, Val Acc: 0.9773
Epoch 14, Train Acc: 0.9885, Val Acc: 0.9782
Epoch 15, Train Acc: 0.9885, Val Acc: 0.9819
Epoch 16, Train Acc: 0.9912, Val Acc: 0.9837
Epoch 17, Train Acc: 0.9885, Val Acc: 0.9819
Epoch 18, Train Acc: 0.9885, Val Acc: 0.9791
Epoch 19, Train Acc: 0.9890, Val Acc: 0.9764
Epoch 20, Train Acc: 0.9909, Val Acc: 0.9801
Epoch 21, Train Acc: 0.9885, Val Acc: 0.9837
Epoch 22, Train Acc: 0.9872, Val Acc: 0.9837
Epoch 23, Train Acc

In [141]:
best_model_state, best_val_acc = train(model = model, lr=1e-3, batch_size=32, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9800, Val Acc: 0.9837
Epoch 2, Train Acc: 0.9794, Val Acc: 0.9719
Epoch 3, Train Acc: 0.9819, Val Acc: 0.9819
Epoch 4, Train Acc: 0.9795, Val Acc: 0.9746
Epoch 5, Train Acc: 0.9786, Val Acc: 0.9837
Epoch 6, Train Acc: 0.9821, Val Acc: 0.9782
Epoch 7, Train Acc: 0.9850, Val Acc: 0.9819
Epoch 8, Train Acc: 0.9827, Val Acc: 0.9810
Epoch 9, Train Acc: 0.9830, Val Acc: 0.9801
Epoch 10, Train Acc: 0.9810, Val Acc: 0.9846
Epoch 11, Train Acc: 0.9802, Val Acc: 0.9819
Epoch 12, Train Acc: 0.9842, Val Acc: 0.9801
Epoch 13, Train Acc: 0.9794, Val Acc: 0.9810
Epoch 14, Train Acc: 0.9811, Val Acc: 0.9782
Epoch 15, Train Acc: 0.9834, Val Acc: 0.9837
Epoch 16, Train Acc: 0.9814, Val Acc: 0.9819
Epoch 17, Train Acc: 0.9858, Val Acc: 0.9791
Epoch 18, Train Acc: 0.9845, Val Acc: 0.9764
Epoch 19, Train Acc: 0.9822, Val Acc: 0.9782
Epoch 20, Train Acc: 0.9813, Val Acc: 0.9810
Epoch 21, Train Acc: 0.9781, Val Acc: 0.9728
Epoch 22, Train Acc: 0.9795, Val Acc: 0.9837
Epoch 23, Train Acc

In [142]:
best_model_state, best_val_acc = train(model = model, lr=1e-4, batch_size=32, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9896, Val Acc: 0.9773
Epoch 2, Train Acc: 0.9888, Val Acc: 0.9837
Epoch 3, Train Acc: 0.9910, Val Acc: 0.9791
Epoch 4, Train Acc: 0.9917, Val Acc: 0.9837
Epoch 5, Train Acc: 0.9914, Val Acc: 0.9801
Epoch 6, Train Acc: 0.9926, Val Acc: 0.9801
Epoch 7, Train Acc: 0.9922, Val Acc: 0.9846
Epoch 8, Train Acc: 0.9918, Val Acc: 0.9846
Epoch 9, Train Acc: 0.9931, Val Acc: 0.9828
Epoch 10, Train Acc: 0.9930, Val Acc: 0.9782
Epoch 11, Train Acc: 0.9914, Val Acc: 0.9819
Epoch 12, Train Acc: 0.9933, Val Acc: 0.9828
Epoch 13, Train Acc: 0.9946, Val Acc: 0.9810
Epoch 14, Train Acc: 0.9931, Val Acc: 0.9855
Epoch 15, Train Acc: 0.9949, Val Acc: 0.9864
Epoch 16, Train Acc: 0.9938, Val Acc: 0.9864
Epoch 17, Train Acc: 0.9941, Val Acc: 0.9864
Epoch 18, Train Acc: 0.9933, Val Acc: 0.9846
Epoch 19, Train Acc: 0.9933, Val Acc: 0.9855
Epoch 20, Train Acc: 0.9933, Val Acc: 0.9855
Epoch 21, Train Acc: 0.9944, Val Acc: 0.9846
Epoch 22, Train Acc: 0.9947, Val Acc: 0.9846
Epoch 23, Train Acc

## SmallModel Optimization

In [143]:
model = SmallModel(input_dim=X_train_t.shape[1], num_classes=6)

### Learning rate tested: 0.0001 ,0.0005, 0.001

In [144]:
best_model_state, best_val_acc = train(model = model, lr=1e-3, batch_size=128, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.8147, Val Acc: 0.9420
Epoch 2, Train Acc: 0.9544, Val Acc: 0.9692
Epoch 3, Train Acc: 0.9658, Val Acc: 0.9737
Epoch 4, Train Acc: 0.9702, Val Acc: 0.9773
Epoch 5, Train Acc: 0.9749, Val Acc: 0.9755
Epoch 6, Train Acc: 0.9792, Val Acc: 0.9692
Epoch 7, Train Acc: 0.9810, Val Acc: 0.9746
Epoch 8, Train Acc: 0.9810, Val Acc: 0.9791
Epoch 9, Train Acc: 0.9848, Val Acc: 0.9801
Epoch 10, Train Acc: 0.9829, Val Acc: 0.9746
Epoch 11, Train Acc: 0.9862, Val Acc: 0.9791
Epoch 12, Train Acc: 0.9843, Val Acc: 0.9801
Epoch 13, Train Acc: 0.9870, Val Acc: 0.9773
Epoch 14, Train Acc: 0.9813, Val Acc: 0.9801
Epoch 15, Train Acc: 0.9851, Val Acc: 0.9746
Epoch 16, Train Acc: 0.9850, Val Acc: 0.9746
Epoch 17, Train Acc: 0.9842, Val Acc: 0.9773
Epoch 18, Train Acc: 0.9875, Val Acc: 0.9828
Epoch 19, Train Acc: 0.9853, Val Acc: 0.9864
Epoch 20, Train Acc: 0.9856, Val Acc: 0.9864
Epoch 21, Train Acc: 0.9885, Val Acc: 0.9746
Epoch 22, Train Acc: 0.9877, Val Acc: 0.9782
Epoch 23, Train Acc

In [145]:
best_model_state, best_val_acc = train(model = model, lr=5e-4, batch_size=128, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9901, Val Acc: 0.9801
Epoch 2, Train Acc: 0.9946, Val Acc: 0.9864
Epoch 3, Train Acc: 0.9944, Val Acc: 0.9828
Epoch 4, Train Acc: 0.9950, Val Acc: 0.9873
Epoch 5, Train Acc: 0.9947, Val Acc: 0.9873
Epoch 6, Train Acc: 0.9965, Val Acc: 0.9837
Epoch 7, Train Acc: 0.9944, Val Acc: 0.9864
Epoch 8, Train Acc: 0.9960, Val Acc: 0.9873
Epoch 9, Train Acc: 0.9968, Val Acc: 0.9846
Epoch 10, Train Acc: 0.9968, Val Acc: 0.9837
Epoch 11, Train Acc: 0.9946, Val Acc: 0.9864
Epoch 12, Train Acc: 0.9952, Val Acc: 0.9846
Epoch 13, Train Acc: 0.9938, Val Acc: 0.9764
Epoch 14, Train Acc: 0.9942, Val Acc: 0.9864
Epoch 15, Train Acc: 0.9958, Val Acc: 0.9846
Epoch 16, Train Acc: 0.9962, Val Acc: 0.9891
Epoch 17, Train Acc: 0.9958, Val Acc: 0.9864
Epoch 18, Train Acc: 0.9963, Val Acc: 0.9873
Epoch 19, Train Acc: 0.9962, Val Acc: 0.9846
Epoch 20, Train Acc: 0.9963, Val Acc: 0.9819
Epoch 21, Train Acc: 0.9962, Val Acc: 0.9855
Epoch 22, Train Acc: 0.9971, Val Acc: 0.9891
Epoch 23, Train Acc

In [146]:
best_model_state, best_val_acc = train(model = model, lr=1e-4, batch_size=128, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9986, Val Acc: 0.9882
Epoch 2, Train Acc: 0.9987, Val Acc: 0.9882
Epoch 3, Train Acc: 0.9986, Val Acc: 0.9873
Epoch 4, Train Acc: 0.9994, Val Acc: 0.9891
Epoch 5, Train Acc: 0.9987, Val Acc: 0.9882
Epoch 6, Train Acc: 0.9990, Val Acc: 0.9891
Epoch 7, Train Acc: 0.9989, Val Acc: 0.9873
Epoch 8, Train Acc: 0.9992, Val Acc: 0.9882
Epoch 9, Train Acc: 0.9990, Val Acc: 0.9882
Epoch 10, Train Acc: 0.9992, Val Acc: 0.9882
Epoch 11, Train Acc: 0.9992, Val Acc: 0.9891
Epoch 12, Train Acc: 0.9992, Val Acc: 0.9891
Epoch 13, Train Acc: 0.9989, Val Acc: 0.9873
Epoch 14, Train Acc: 0.9995, Val Acc: 0.9873
Epoch 15, Train Acc: 0.9997, Val Acc: 0.9873
Epoch 16, Train Acc: 0.9997, Val Acc: 0.9882
Epoch 17, Train Acc: 0.9997, Val Acc: 0.9882
Epoch 18, Train Acc: 0.9997, Val Acc: 0.9882
Epoch 19, Train Acc: 0.9994, Val Acc: 0.9882
Epoch 20, Train Acc: 0.9992, Val Acc: 0.9891
Epoch 21, Train Acc: 0.9995, Val Acc: 0.9900
Epoch 22, Train Acc: 0.9997, Val Acc: 0.9891
Epoch 23, Train Acc

### Batch Size tested: 32, 64, 256

In [147]:
best_model_state, best_val_acc = train(model = model, lr=1e-4, batch_size=256, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9995, Val Acc: 0.9900
Epoch 2, Train Acc: 0.9995, Val Acc: 0.9900
Epoch 3, Train Acc: 0.9995, Val Acc: 0.9891
Epoch 4, Train Acc: 0.9995, Val Acc: 0.9900
Epoch 5, Train Acc: 0.9998, Val Acc: 0.9900
Epoch 6, Train Acc: 0.9998, Val Acc: 0.9900
Epoch 7, Train Acc: 0.9997, Val Acc: 0.9882
Epoch 8, Train Acc: 0.9995, Val Acc: 0.9891
Epoch 9, Train Acc: 0.9997, Val Acc: 0.9900
Epoch 10, Train Acc: 0.9998, Val Acc: 0.9900
Epoch 11, Train Acc: 0.9998, Val Acc: 0.9900
Epoch 12, Train Acc: 0.9990, Val Acc: 0.9891
Epoch 13, Train Acc: 0.9998, Val Acc: 0.9891
Epoch 14, Train Acc: 0.9998, Val Acc: 0.9900
Epoch 15, Train Acc: 0.9995, Val Acc: 0.9909
Epoch 16, Train Acc: 1.0000, Val Acc: 0.9900
Epoch 17, Train Acc: 0.9997, Val Acc: 0.9918
Epoch 18, Train Acc: 0.9995, Val Acc: 0.9900
Epoch 19, Train Acc: 0.9995, Val Acc: 0.9891
Epoch 20, Train Acc: 0.9997, Val Acc: 0.9918
Epoch 21, Train Acc: 0.9997, Val Acc: 0.9918
Epoch 22, Train Acc: 0.9997, Val Acc: 0.9909
Epoch 23, Train Acc

In [148]:
best_model_state, best_val_acc = train(model = model, lr=1e-4, batch_size=64, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9994, Val Acc: 0.9909
Epoch 2, Train Acc: 0.9994, Val Acc: 0.9900
Epoch 3, Train Acc: 0.9987, Val Acc: 0.9891
Epoch 4, Train Acc: 1.0000, Val Acc: 0.9882
Epoch 5, Train Acc: 0.9997, Val Acc: 0.9900
Epoch 6, Train Acc: 0.9995, Val Acc: 0.9891
Epoch 7, Train Acc: 0.9998, Val Acc: 0.9900
Epoch 8, Train Acc: 0.9994, Val Acc: 0.9900
Epoch 9, Train Acc: 0.9997, Val Acc: 0.9873
Epoch 10, Train Acc: 1.0000, Val Acc: 0.9900
Epoch 11, Train Acc: 1.0000, Val Acc: 0.9873
Epoch 12, Train Acc: 0.9995, Val Acc: 0.9882
Epoch 13, Train Acc: 0.9995, Val Acc: 0.9864
Epoch 14, Train Acc: 0.9997, Val Acc: 0.9909
Epoch 15, Train Acc: 0.9998, Val Acc: 0.9891
Epoch 16, Train Acc: 0.9998, Val Acc: 0.9891
Epoch 17, Train Acc: 0.9995, Val Acc: 0.9891
Epoch 18, Train Acc: 0.9995, Val Acc: 0.9909
Epoch 19, Train Acc: 0.9995, Val Acc: 0.9918
Epoch 20, Train Acc: 0.9997, Val Acc: 0.9873
Epoch 21, Train Acc: 0.9998, Val Acc: 0.9873
Epoch 22, Train Acc: 0.9998, Val Acc: 0.9891
Epoch 23, Train Acc

In [149]:
best_model_state, best_val_acc = train(model = model, lr=1e-4, batch_size=32, epochs=30)
print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Train Acc: 0.9992, Val Acc: 0.9873
Epoch 2, Train Acc: 0.9981, Val Acc: 0.9891
Epoch 3, Train Acc: 0.9982, Val Acc: 0.9882
Epoch 4, Train Acc: 0.9987, Val Acc: 0.9882
Epoch 5, Train Acc: 0.9986, Val Acc: 0.9891
Epoch 6, Train Acc: 0.9987, Val Acc: 0.9864
Epoch 7, Train Acc: 0.9990, Val Acc: 0.9900
Epoch 8, Train Acc: 0.9994, Val Acc: 0.9873
Epoch 9, Train Acc: 0.9998, Val Acc: 0.9900
Epoch 10, Train Acc: 0.9990, Val Acc: 0.9864
Epoch 11, Train Acc: 0.9989, Val Acc: 0.9891
Epoch 12, Train Acc: 0.9997, Val Acc: 0.9882
Epoch 13, Train Acc: 0.9992, Val Acc: 0.9927
Epoch 14, Train Acc: 0.9995, Val Acc: 0.9882
Epoch 15, Train Acc: 0.9997, Val Acc: 0.9891
Epoch 16, Train Acc: 0.9994, Val Acc: 0.9918
Epoch 17, Train Acc: 0.9997, Val Acc: 0.9900
Epoch 18, Train Acc: 0.9987, Val Acc: 0.9900
Epoch 19, Train Acc: 0.9998, Val Acc: 0.9900
Epoch 20, Train Acc: 0.9984, Val Acc: 0.9873
Epoch 21, Train Acc: 0.9981, Val Acc: 0.9918
Epoch 22, Train Acc: 0.9989, Val Acc: 0.9882
Epoch 23, Train Acc

# Prediction and Metrics on the Test set

In [155]:
model.load_state_dict(best_model_state)

<All keys matched successfully>

In [156]:
model.eval()

with torch.no_grad():
    outputs = model(X_test_t)
    preds = torch.argmax(outputs, dim=1)

In [157]:
y_pred = preds.cpu().numpy()
y_true = y_test_encoded

In [158]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average='weighted'))
print("Recall:", recall_score(y_true, y_pred, average='weighted'))
print("F1-score:", f1_score(y_true, y_pred, average='weighted'))

Accuracy: 0.9636918900576857
Precision: 0.9648019649950331
Recall: 0.9636918900576857
F1-score: 0.9636199959369987
